# WWW::HorizonsEphemerisSystem basic usage

Anton Antonov  
April 2026

---

## Introduction

This notebook has examples for the basic usage of the sub `horizons-ephemeris-data` provided by the package ["WWW::HorizonsEphemerisSystem"](https://raku.land/zef:antononcube/WWW::HorizonsEphemerisSystem).

---

## Setup

In [34]:
use WWW::HorizonsEphemerisSystem;

----

## Basic usage

Cartesian position of Mars with default parameters.

In [35]:
my $mars-default = horizons-ephemeris-data(
    'state',
    '499',
    'position',
    :modifier<data>,
);

[{Calendar Date (TDB) => A.D. 2026-Apr-13 12:09:54.0000, Date => 2461144.006875000, Vx => 6.118226785847199, Vy => 25.74349198114207, Vz => 0.3894750521643466, X => 201777945.5627485, Y => -45030435.53253341, Z => -5865330.365737507}]

Cartesian position of Mars for a specific date.

In [36]:
my $date = "{.year}-{.month.fmt('%02s')}-{.day.fmt('%02s')}" with DateTime.now(:12hours);

my $mars-at-date = horizons-ephemeris-data(
    'state',
    ['499', {
        dates => $date,
    }],
    'position',
    :modifier<data>,
);

[{Calendar Date (TDB) => A.D. 2026-Apr-13 00:00:00.0000, Date => 2461143.500000000, Vx => 6.250624479100147, Vy => 25.71413840127359, Vz => 0.3856135842699668, X => 201507104.3176457, Y => -46157205.99005968, Z => -5882302.526465395}]

Azimuth/Elevation of the Moon from Moscow (lat, lon) for next 12 hours.

In [39]:
my @dates = do with DateTime.now { ($_, $_.later(:12hours), $_.later(minutes => 12*60 + 10))};
@dates = @dates.map({ "{.year}-{.month.fmt('%02s')}-{.day.fmt('%02s')} {.hh-mm-ss}" });
say (:@dates);

my $tycho-from-moscow = horizons-ephemeris-data(
    'observer',
    {
        target => '301',
        center => (55.7505, 37.6175),
        :@dates,
    },
    <azimuth elevation>,
    :modifier<data>,
);

dates => [2026-04-13 08:10:59 2026-04-13 20:10:59 2026-04-13 20:20:59]


[{Azi_(a-app) => *, Date_________JDUT => 2461143.840960648, Elev_(a-app) => m} {Azi_(a-app) => , Date_________JDUT => 2461144.340960648, Elev_(a-app) => } {Azi_(a-app) => , Date_________JDUT => 2461144.347905092, Elev_(a-app) => }]

Here is a range of dates from now to 4 hours from now with step 30 min:

In [43]:
my $start = DateTime.now;
my $end = $start.later(:4hours);
my $current = $start;
my @dates;

while $current <= $end {
    @dates.push($current);
    $current = $current.later(:30minutes);
}

#@dates = @dates».Str.map({ $_.substr(^19).subst('T', ' ') });
@dates = @dates.map({ "{.year}-{.month.fmt('%02s')}-{.day.fmt('%02s')} {.hh-mm-ss}" });


deduce-type(@dates);

Vector(Atom((Str)), 9)

Orbital elements of Saturn moon Pandora for a specific list of date-times.

In [44]:
my $pandora-elements = horizons-ephemeris-data(
    'orbital-elements',
    ['617', {
        center => '699',
        :@dates
    }],
    'all',
    :modifier<association>,
);

deduce-type($pandora-elements)

Assoc(Atom((Str)), Struct([ApoapsisDistance, AscendingNodeLongitude, Calendar Date (TDB), Date, Eccentricity, Inclination, MeanAnomaly, MeanMotion, OrbitalPeriod, PeriapsisDate, PeriapsisDistance, PerifocusArgument, SemiMajorAxis, TrueAnomaly], [Num, Num, Str, Str, Num, Num, Num, Num, Num, Str, Num, Num, Num, Num]), 9)

In [45]:
#%html
$pandora-elements.values
==> to-html(field-names => ['Calendar Date (TDB)', 'Date', |$pandora-elements.values.head.keys.grep({ $_ ∉ ['Calendar Date (TDB)', 'Date'] }).sort])

Calendar Date (TDB),Date,ApoapsisDistance,AscendingNodeLongitude,Eccentricity,Inclination,MeanAnomaly,MeanMotion,OrbitalPeriod,PeriapsisDate,PeriapsisDistance,PerifocusArgument,SemiMajorAxis,TrueAnomaly
A.D. 2026-Apr-13 09:42:03.0000,2461143.904201389,142811.9015095493,169.6267970523802,0.003251041537768731,28.03892791326052,295.9990077395854,0.006570354246968413,54791.56624866998,2.461144016942985E+06,141886.3357146041,216.5244928630105,142349.1186120767,295.6635708651363
A.D. 2026-Apr-13 08:42:03.0000,2461143.862534722,142566.8777354964,169.6269390505598,0.001538110800280129,28.0388440372029,288.6318237570555,0.006570436482751401,54790.88047575925,2.461143988252482E+06,142128.9839558338,200.0271463868201,142347.9308456651,288.4647041321576
A.D. 2026-Apr-13 12:12:03.0000,2461144.008368055,143327.0521867028,169.6268459364473,0.006833811170459676,28.03934551966565,323.2647573492283,0.00657000035841525,54794.51755872287,2.461144073082880E+06,141381.4083289067,248.9219929204588,142354.2302578048,322.79315037178
A.D. 2026-Apr-13 10:42:03.0000,2461143.945868055,143039.8715321997,169.6267071333511,0.004839637837948538,28.03908088168261,306.4825699366416,0.006570227832742674,54792.62046377495,2.461144040144040E+06,141662.0174975161,229.9058606263561,142350.9445148579,306.035060453
A.D. 2026-Apr-13 09:12:03.0000,2461143.883368055,142690.7672571811,169.6268695171633,0.002404920502939984,28.03887646345816,291.4914517417203,0.006570401881871825,54791.16901406958,2.461144004049107E+06,142006.0939385688,209.0997404696968,142348.4305978749,291.2347470133694
A.D. 2026-Apr-13 10:12:03.0000,2461143.925034722,142928.7168843453,169.6267381507203,0.004065677746995614,28.03899739965907,301.1029077781408,0.006570295454768903,54792.05653357674,2.461144028786420E+06,141771.2186917027,223.3530019828166,142349.967788024,300.7029450381024
A.D. 2026-Apr-13 08:12:03.0000,2461143.841701389,142443.3844808842,169.6269897132378,0.0006726147204582796,28.03882844816273,293.9271683151715,0.00657045666399052,54790.71218489044,2.461143958090840E+06,142251.8942455678,182.7996484084032,142347.639363226,293.8566921062186
A.D. 2026-Apr-13 11:12:03.0000,2461143.966701389,143144.1037700744,169.6267136921215,0.005564332123938692,28.03917192090347,312.0069781167116,0.006570154073347584,54793.23558946237,2.461144051246558E+06,141559.9160467929,236.3140941963987,142352.0099084337,311.5309721960772
A.D. 2026-Apr-13 11:42:03.0000,2461143.987534722,143240.2102298207,169.6267611805821,0.006231613819212317,28.0392627367271,317.6125243601211,0.006570077182356363,54793.87684619037,2.461144062205958E+06,141466.0308832843,242.6413187724009,142353.1205565525,317.1283475539266


Using date range spec (the third element is automatically recognized to be a time step):

In [47]:
my $start = DateTime.now;
my $end = $start.later(:12hours);
my @date-range = $start, $end, "10 m";
@date-range .= map({ $_ ~~ DateTime ?? "{.year}-{.month.fmt('%02s')}-{.day.fmt('%02s')} {.hh-mm-ss}" !! $_ });
say (:@date-range);

my $pandora-elements-range = horizons-ephemeris-data(
    'orbital-elements',
    ['617', {
        center => '699',
        dates => @date-range
    }],
    'all',
    :modifier<association>,
);

deduce-type($pandora-elements)

date-range => [2026-04-13 08:13:47 2026-04-13 20:13:47 10 m]


Assoc(Atom((Str)), Struct([ApoapsisDistance, AscendingNodeLongitude, Calendar Date (TDB), Date, Eccentricity, Inclination, MeanAnomaly, MeanMotion, OrbitalPeriod, PeriapsisDate, PeriapsisDistance, PerifocusArgument, SemiMajorAxis, TrueAnomaly], [Num, Num, Str, Str, Num, Num, Num, Num, Num, Str, Num, Num, Num, Num]), 9)

Show a sample:

In [48]:
#%html
$pandora-elements-range.values
==> { .pick(6).sort(*<Date>)}()
==> to-html(field-names => ['Calendar Date (TDB)', 'Date', |$pandora-elements-range.values.head.keys.grep({ $_ ∉ ['Calendar Date (TDB)', 'Date'] }).sort])

Calendar Date (TDB),Date,ApoapsisDistance,AscendingNodeLongitude,Eccentricity,Inclination,MeanAnomaly,MeanMotion,OrbitalPeriod,PeriapsisDate,PeriapsisDistance,PerifocusArgument,SemiMajorAxis,TrueAnomaly
A.D. 2026-Apr-13 10:03:47.0000,2461143.919293982,142897.037701059,169.6267521394294,0.00384488405014754,28.03897663287258,299.6606019598365,0.006570312642339531,54791.91320061942,2.461144025586132E+06,141802.4013686937,221.5072554540583,142349.7195348763,299.2768307003518
A.D. 2026-Apr-13 11:03:47.0000,2461143.960960648,143116.1430589464,169.6267078147381,0.005370031413850942,28.03914651460645,310.4746161860289,0.006570174843068642,54793.06237638262,2.461144048204972E+06,141587.2767509366,234.5583378463048,142351.7099049415,310.0044726686722
A.D. 2026-Apr-13 13:33:47.0000,2461144.065127315,143508.5513632716,169.6271692954522,0.008089978500804257,28.039487364192,338.7674290248311,0.006569816425572294,54796.05162158554,2.461144102532827E+06,141205.2230493904,265.9305889240828,142356.887206331,338.4285046035002
A.D. 2026-Apr-13 14:43:47.0000,2461144.113738426,143590.3713280922,169.6273620392018,0.008654817190899921,28.03951005869203,352.0807128815113,0.006569719568020504,54796.85948124422,2.461144127690074E+06,141126.2014395685,280.4621494672845,142358.2863838304,351.9425897040925
A.D. 2026-Apr-13 17:23:47.0000,2461144.224849537,143497.2879636543,169.6271271218996,0.00801083019098444,28.03964298269554,22.52399200626562,0.00656981615723054,54796.05385971035,2.461144185168921E+06,141216.4942016947,313.6648752660486,142356.8910826745,22.87891985802488
A.D. 2026-Apr-13 19:33:47.0000,2461144.315127315,143160.5271489625,169.6271719444801,0.005678703175797425,28.04002006485635,47.09518313961529,0.006570144268423806,54793.3173598888,2.461144232163667E+06,141543.7759189675,340.8036895561604,142352.151533965,47.57414314654995


Here is summary for some of the columns:

In [51]:
my @field-names = <Date Eccentricity Inclination MeanMotion>;
sink records-summary($pandora-elements-range.values.map({ $_.grep(*.key ∈ @field-names) })».Hash, :@field-names)

+-------------------------+---------------------------------+------------------------------+---------------------------------+
| Date                    | Eccentricity                    | Inclination                  | MeanMotion                      |
+-------------------------+---------------------------------+------------------------------+---------------------------------+
| 2461143.940127315 => 1  | Min    => 0.0007213713804904453 | Min    => 28.03882896745138  | Min    => 0.006569699000959045  |
| 2461143.870682871 => 1  | 1st-Qu => 0.005045889580487024  | 1st-Qu => 28.039161883099354 | 1st-Qu => 0.006569772753614565  |
| 2461144.155405093 => 1  | Mean   => 0.0063846204070341864 | Mean   => 28.039437852168987 | Mean   => 0.0065700062932007594 |
| 2461144.308182871 => 1  | Median => 0.007050070586259509  | Median => 28.03950794494576  | Median => 0.006569971031831149  |
| 2461144.044293982 => 1  | 3rd-Qu => 0.008318220893375793  | 3rd-Qu => 28.039631752717007 | 3rd-Qu => 0.006570